In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne

import mne
import numpy as np
import pandas as pd
import os

In [ ]:
# Corrected version without preload
evoked = mne.read_evokeds(evoked_path, condition=0)

# Choose one region and average across its channels
region_channels =  ['E48', 'E43', 'E44', 'E38', 'E32', 'E128', 'E25', 'E22', 'E26', 'E23', 'E27', 'E33', 'E34', 'E35', 'E40', 'E39', 'E28', 'E24', 'E20']   # Example: Left Anterior subset
evoked_region = evoked.copy().pick_channels(region_channels)
data = evoked_region.data  # shape: (n_channels, n_times)

# Average across channels to get a single time series
avg_signal = data.mean(axis=0)

# Prepare overlapping segments (windows of 200 ms with 50% overlap)
sfreq = evoked.info['sfreq']
window_size = int(0.2 * sfreq)  # 200 ms
step_size = int(0.1 * sfreq)    # 50% overlap
segments = [
    avg_signal[i:i+window_size]
    for i in range(0, len(avg_signal) - window_size + 1, step_size)
]
X = np.stack(segments)  # shape: (n_windows, window_size)

# Display the shape of X for inspection
X.shape


In [ ]:
# === STEP 1: Add repo code path ===
import sys
sys.path.append('/content/ds-hdp-hmm/code')

# === STEP 2: Import functions from gibbs_gaussian.py ===
from gibbs_gaussian import init_gibbs_full_bayesian, sample_zw, decre_K, sample_kappa, sample_m, sample_beta, sample_alpha, sample_gamma, sample_rho

# === STEP 3: Use EEG signal as yt ===
yt = avg_signal

# === STEP 4: Initialize the model ===
rho0, rho1, alpha0, gamma0, sigma0, mu0, sigma0_pri, K, zt, wt, beta_vec, beta_new, kappa_vec, kappa_new, n_mat, ysum, ycnt = init_gibbs_full_bayesian(
    p='inf',
    v0_range=[0.01, 0.99],
    v1_range=[0.01, 0.2],
    alpha0_a_pri=1,
    alpha0_b_pri=0.01,
    gamma0_a_pri=1,
    gamma0_b_pri=0.01,
    sigma0=np.std(yt),
    mu0=np.mean(yt),
    sigma0_pri=np.std(yt),
    T=len(yt),
    yt=yt
)

# === STEP 5: Run a few Gibbs sampling iterations ===
for i in range(10):  # increase for convergence
    zt, wt, n_mat, ysum, ycnt, beta_vec, kappa_vec, beta_new, kappa_new, K = sample_zw(
        zt, wt, yt, n_mat, ysum, ycnt,
        beta_vec, beta_new, kappa_vec, kappa_new,
        alpha0, gamma0, sigma0, mu0, sigma0_pri,
        rho0, rho1, K
    )
    zt, n_mat, ysum, ycnt, beta_vec, K = decre_K(zt, n_mat, ysum, ycnt, beta_vec)
    kappa_vec, kappa_new, num_1_vec, num_0_vec = sample_kappa(zt, wt, rho0, rho1, K)
    m_mat = sample_m(n_mat, beta_vec, alpha0, K)
    beta_vec, beta_new = sample_beta(m_mat, gamma0)
    alpha0 = sample_alpha(m_mat, n_mat, alpha0, 1, 0.01)
    gamma0 = sample_gamma(K, m_mat, gamma0, 1, 0.01)
    rho0, rho1, _ = sample_rho([0.01,0.99],[0.01,0.2],10,10,K,num_1_vec,num_0_vec,'inf')

# === STEP 6: Output hidden states ===
print("Inferred hidden states z(t):", zt)


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from math import log2
from scipy.stats import norm
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve

# === STEP 1: 定义 z(t) summary 特征函数 ===

def compute_entropy(z_seq):
    counts = Counter(z_seq)
    probs = [v / len(z_seq) for v in counts.values()]
    return -sum(p * log2(p) for p in probs if p > 0)

def extract_features_from_zt(z_seq):
    return {
        'z_entropy': compute_entropy(z_seq),
        'z_switches': np.sum(np.diff(z_seq) != 0),
        'z_num_states': len(set(z_seq)),
        'z_mode_state': Counter(z_seq).most_common(1)[0][0],
        'z_mean_state': np.mean(z_seq),
    }

# === STEP 2: 导入原始 df_results ===
df_results = pd.read_csv('/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Presence_Table.csv')

# === STEP 3: 你的全部 z(t)（一个 trial 对应一个）===
all_zts = [zt] * len(df_results)

# === STEP 4: 提取每个 trial 的 z(t) 特征 ===
z_feature_rows = [extract_features_from_zt(z_seq) for z_seq in all_zts]
df_z_features = pd.DataFrame(z_feature_rows)

# === STEP 5: 合并进原始表 ===
df_results = pd.concat([df_results.reset_index(drop=True), df_z_features], axis=1)

# === STEP 6: 将 z(t) 特征用于修改模型参数 ===


# 使用 sticky-HDP-HMM 的状态切换数 z_switches 来微调 mu_c
def get_mu_c(row):
    base_mu = 0.96 if row['Condition'] == 'Normal' else 1.15
    return base_mu + 0.01 * row['z_switches']  # 每切换 1 次状态，mu_c 增加 10ms

def get_sigma_p(row):
    # 让 sigma_p 随 entropy 增加
    return 0.05 + 0.01 * row['z_entropy']

df_results['mu_c'] = df_results.apply(get_mu_c, axis=1)
df_results['sigma_p'] = df_results.apply(get_sigma_p, axis=1)


# === STEP 7: 重新计算 P_ERP ===

t_star = 0.96
delta = 0.05
sigma_l = 0.03
theta = 0.6

def compute_p_erp(mu_c, sigma_p, t_star=0.96, sigma_l=0.03, delta=0.05):
    sigma_post_sq = 1 / (1/sigma_p**2 + 1/sigma_l**2)
    mu_post = sigma_post_sq * (mu_c/sigma_p**2 + t_star/sigma_l**2)
    sigma_post = sigma_post_sq**0.5
    p_erp = norm(mu_post, sigma_post).cdf(t_star + delta) - norm(mu_post, sigma_post).cdf(t_star - delta)
    return p_erp

df_results['P_ERP'] = df_results.apply(
    lambda row: compute_p_erp(row['mu_c'], row['sigma_p']), axis=1
)

df_results['ERP_Predict'] = df_results['P_ERP'] > theta

# === STEP 8: 模型性能评估 ===

acc = accuracy_score(df_results['ERP_Present'], df_results['ERP_Predict'])
print(f"Accuracy: {acc:.2f}")
print(confusion_matrix(df_results['ERP_Present'], df_results['ERP_Predict']))

y_true = df_results['ERP_Present']
y_score = df_results['P_ERP']

fpr, tpr, _ = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Sticky-HDP-HMM Modulated ERP Prediction")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import f1_score

best_theta = 0
best_f1 = 0
for theta in np.linspace(0, 1, 100):
    preds = df_results['P_ERP'] > theta
    f1 = f1_score(df_results['ERP_Present'], preds)
    if f1 > best_f1:
        best_f1 = f1
        best_theta = theta

print(f"Best θ: {best_theta:.2f} with F1 = {best_f1:.2f}")


In [ ]:
import seaborn as sns
sns.histplot(data=df_results, x='P_ERP', hue='ERP_Present', bins=30, kde=True)
plt.title("Distribution of P_ERP by true ERP presence")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve, f1_score

# === STEP 1: 加载 ERP 数据表格 ===
df_results = pd.read_csv('/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Presence_Table.csv')
N = len(df_results)

# === STEP 2: 填入 sticky-HDP-HMM z(t) ===
zt = np.array([
    0,15,15,15,15,15,15,19,19,9,9,9,13,13,13,14,14,14,14,20,20,20,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,
    11,11,11,11,11,11,11,4,4,4,4,2,2,8,8,8,8,8,8,8,8,8,18,4,4,4,4,16,16,16,16,5,5,9,9,9,9,9,9,6,6,6,6,6,6,6,6,6,6,6,
    6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,6,21,12,12,12,12,22,16,16,16,16,16,23,23,10,10,10,10,10,3,3,3,3,3,3,3,3,3,3,3,
    3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,9,16,16,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,
    7,7,7,7,7,7,7,7,7,7,7,7,7,7,7,23,4,4,4,16,16,17,17
])

# === STEP 3: 划分成每个 trial 的状态序列 ===
def count_state_switches(z_array):
    return np.sum(np.diff(z_array) != 0)

segments = np.array_split(zt, N)  # 自动平均切分为 N 段
df_results['z_switches'] = [count_state_switches(seg) for seg in segments]

# === STEP 4: 定义参数 ===
t_star = 0.96
delta = 0.05
sigma_l = 0.03
base_sigma_p = 0.05

# === STEP 5: 计算 mu_c，加入 z-switch 调制 ===
def get_mu_c(row):
    base_mu = 0.96 if row['Condition'] == 'Normal' else 1.15
    return base_mu + 0.07 * (1 / (1 + np.exp(-0.8 * (row['z_switches'] - 3))) - 0.5)

df_results['mu_c'] = df_results.apply(get_mu_c, axis=1)
df_results['sigma_p'] = base_sigma_p

# === STEP 6: 计算 P_ERP 概率 ===
def compute_p_erp(mu_c, sigma_p, t_star=0.96, sigma_l=0.03, delta=0.05):
    sigma_post_sq = 1 / (1/sigma_p**2 + 1/sigma_l**2)
    mu_post = sigma_post_sq * (mu_c/sigma_p**2 + t_star/sigma_l**2)
    sigma_post = sigma_post_sq**0.5
    return norm(mu_post, sigma_post).cdf(t_star + delta) - norm(mu_post, sigma_post).cdf(t_star - delta)

df_results['P_ERP'] = df_results.apply(
    lambda row: compute_p_erp(row['mu_c'], row['sigma_p']), axis=1
)

# === STEP 7: 寻找最优阈值 θ ===
best_theta = 0
best_f1 = 0
for theta_candidate in np.linspace(0, 1, 100):
    preds = df_results['P_ERP'] > theta_candidate
    f1 = f1_score(df_results['ERP_Present'], preds)
    if f1 > best_f1:
        best_f1 = f1
        best_theta = theta_candidate

print(f" Best θ: {best_theta:.2f} with F1 = {best_f1:.2f}")

# === STEP 8: 应用最优 θ 进行预测 ===
df_results['ERP_Predict'] = df_results['P_ERP'] > best_theta

# === STEP 9: 打印评估指标 ===
acc = accuracy_score(df_results['ERP_Present'], df_results['ERP_Predict'])
print(f"🎯 Accuracy: {acc:.2f}")
print(confusion_matrix(df_results['ERP_Present'], df_results['ERP_Predict']))

# === STEP 10: ROC 曲线绘图 ===
y_true = df_results['ERP_Present']
y_score = df_results['P_ERP']
fpr, tpr, _ = roc_curve(y_true, y_score)
auc = roc_auc_score(y_true, y_score)

plt.plot(fpr, tpr, label=f'AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Sticky-HDP-HMM Modulated ERP Prediction")
plt.legend()
plt.show()

# === STEP 11: P_ERP 分布图 ===
sns.histplot(data=df_results, x='P_ERP', hue='ERP_Present', bins=40, kde=True, palette='Set2')
plt.title("Distribution of P_ERP by true ERP presence")
plt.show()
